# Prompting and Efficiently Fine-Tuning a Large Language Model (v2)

This notebook covers two broad approaches for working with Large Language Models (LLMs):

1. **Prompting** — Steering a pre-trained model through carefully designed inputs, without changing any weights.
2. **Fine-Tuning** — Updating model weights on a task-specific dataset to specialise behaviour.

---

## Why Fine-Tune?

Pre-trained LLMs are trained on broad internet text and already possess impressive general knowledge. However, they may:

- Produce output in the wrong format for a downstream system
- Lack domain-specific knowledge (medical, legal, etc.)
- Not follow instructions reliably
- Generate responses in the wrong language or style

Fine-tuning addresses these gaps by training on curated examples.

---

## Fine-Tuning Strategies

### Supervised Fine-Tuning (SFT)
The model is trained directly on `(instruction, response)` pairs. The loss is standard cross-entropy over the response tokens. This is the simplest and most common fine-tuning approach.

### Reinforcement Learning from Human Feedback (RLHF)
Human annotators rank model outputs. A separate *reward model* is trained on these rankings, and the LLM is then optimised to maximise the reward using Proximal Policy Optimisation (PPO). This is how ChatGPT was trained, but it is complex and expensive.

### Direct Preference Optimisation (DPO)
A simpler alternative to RLHF that skips the reward model entirely. The LLM is trained directly on `(prompt, chosen_response, rejected_response)` triples, making it much easier to implement while achieving similar alignment quality.

---

In this notebook we focus on **SFT** using the modern **QLoRA** approach.

## Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | Model loading, tokenisation, generation |
| `peft` | LoRA adapter creation and management |
| `trl` | `SFTTrainer` and `SFTConfig` for instruction fine-tuning |
| `datasets` | Dataset loading from the HuggingFace hub |
| `bitsandbytes` | 4-bit quantisation (BnB) |
| `accelerate` | Multi-GPU / mixed-precision training backend |

> **Note for HPC users**: On SLURM clusters, pre-install these in your virtual environment or Singularity container rather than running pip inside the notebook.

In [ ]:
!pip install -q -U trl transformers accelerate
!pip install -q -U datasets bitsandbytes peft

In [ ]:
# Authenticate to access gated models (e.g. Llama-3)
# You need a HuggingFace account and to accept the model licence at huggingface.co/meta-llama
!huggingface-cli login

## Loading the Model

### Why Llama-3?

We use **Meta Llama-3-8B-Instruct** here instead of the Llama-2 model used in the previous version. Llama-3 offers:

- A much larger vocabulary (128k tokens vs 32k) — better multilingual and code coverage
- Improved instruction following out-of-the-box
- A well-defined chat template (using special `<|begin_of_text|>`, `<|eot_id|>` tokens)
- Better performance on standard benchmarks at the same parameter count

For the 8B model you need roughly **6 GB of VRAM** with 4-bit quantisation (vs ~26 GB at full precision).

## Quantisation: How to Fit a Large Model in Memory

Neural network weights are floating-point numbers. By default, modern models use **bfloat16** (2 bytes per parameter). An 8B parameter model therefore needs **~16 GB** of memory just to store weights — before considering activations and optimizer states during training.

**Quantisation** converts weights to a lower bit-width representation:

| Precision | Bytes/param | Memory for 8B model |
|---|---|---|
| float32 | 4 | ~32 GB |
| bfloat16 | 2 | ~16 GB |
| int8 | 1 | ~8 GB |
| nf4 (4-bit) | 0.5 | ~4 GB |

### NF4 (NormalFloat 4-bit)
NF4 is a data type introduced in the **QLoRA paper** (Dettmers et al., 2023). It is theoretically optimal for normally-distributed weights: the 16 quantisation levels are spaced to match the cumulative distribution of the weights, minimising quantisation error.

### Double Quantisation
The quantisation constants themselves (one per small block of weights) are also quantised — saving an additional ~0.37 bits per parameter.

### Compute dtype
Even though weights are stored in 4-bit, computations are performed in `bfloat16` (upcast on the fly), maintaining numerical stability.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Store weights in 4-bit
    bnb_4bit_quant_type="nf4",      # Use NormalFloat 4-bit (optimal for normally-distributed weights)
    bnb_4bit_use_double_quant=True, # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for actual matrix multiplications
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",       # Automatically distribute layers across available GPUs
    trust_remote_code=True,
)
model.config.use_cache = False  # Disable KV cache during training (not needed, saves memory)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Llama-3 does not have a dedicated pad token — use eos_token as pad
# padding_side='right' avoids overflow issues in half-precision training
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Prompt Engineering

Before fine-tuning, it is worth exploring what the model can already do through careful prompting. Effective prompting can often replace fine-tuning for many tasks — saving significant compute.

### Chat Templates

Modern instruction-tuned models expect input in a specific structured format called a **chat template**. Each model family defines its own template. Using the wrong format leads to degraded performance.

The HuggingFace tokenizer provides `apply_chat_template()` to handle this automatically:

In [ ]:
from transformers import pipeline

# Create the pipeline without generation parameters — the pipeline already
# manages its own GenerationConfig internally, so passing one at construction
# causes a conflict. Instead, pass generation parameters at call time.
generator = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
)

def chat(user_message, system_message="You are a helpful assistant.",
         temperature=0.1, max_new_tokens=300, repetition_penalty=1.1):
    """Format a message using the model's chat template and generate a response."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]
    response = generator(
        messages,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        repetition_penalty=repetition_penalty,
        do_sample=True,
    )
    return response[0]["generated_text"][-1]["content"]

## Prompting Strategies

The following sections illustrate the main prompting techniques, from simplest to most sophisticated.

### 1. Zero-Shot Prompting

Ask the model to perform a task with no examples — relying entirely on pre-trained knowledge.

**When to use:** The task is common and well-represented in the training data (e.g., translation, summarisation, simple classification).

**Limitations:** Can fail on domain-specific or unusual tasks.

In [ ]:
response = chat(
    user_message="Classify the sentiment of this text as positive, neutral, or negative:\n"
                 "'I think the vacation was okay.'"
)
print(response)

### 2. Few-Shot Prompting

Provide a handful of labelled examples (**shots**) in the prompt before the actual query. The model performs **in-context learning** — it infers the task format and label space from the examples without any weight updates.

**Key insight:** The examples do not fine-tune the model; they influence only the attention pattern for that single forward pass.

**When to use:** The task is novel or the zero-shot output format is inconsistent.

In [ ]:
few_shot_prompt = """Classify each review as positive, neutral, or negative.

Review: 'This restaurant is the best I've ever been to. Delicious food, friendly staff.'
Sentiment: positive

Review: 'I was disappointed — the product broke within a week.'
Sentiment: negative

Review: 'The movie was okay, not great but not terrible either.'
Sentiment: neutral

Review: 'I absolutely love the new Spider-Man film. Incredibly well done!'
Sentiment:"""

response = chat(user_message=few_shot_prompt)
print(response)

### 3. System Prompts

The `system` role in a chat template lets you define the model's **persona, rules, and constraints** that persist for the whole conversation. Unlike user instructions, system prompts are typically hidden from end users.

**Use cases:** Chatbots with specific personalities, content-filtered assistants, domain-constrained tools.

In [ ]:
system = (
    "You are a grumpy pirate who answers every question correctly "
    "but complains about having to do so. Always end with 'Arrr!'"
)

response = chat(
    user_message="What is the capital of France?",
    system_message=system
)
print(response)

### 4. Chain-of-Thought (CoT) Prompting

For **reasoning-intensive tasks** (arithmetic, logic, multi-step problems), LLMs perform significantly better when instructed to *reason step by step* before giving the final answer. This is called **Chain-of-Thought** prompting (Wei et al., 2022).

The key insight is that generating intermediate reasoning steps provides additional context tokens that guide the model toward the correct answer — essentially using the output space as a scratchpad.

**Variants:**
- **Zero-shot CoT:** Simply append *"Let's think step by step."* to the prompt.
- **Few-shot CoT:** Include worked examples that show reasoning steps.
- **Self-consistency:** Sample multiple CoT paths and take a majority vote.

In [ ]:
# Without CoT — model may jump to wrong answer
problem = (
    "A store sells apples for €1.20 each. A customer buys 3 apples and pays with a €5 note. "
    "How much change does the customer receive?"
)

print("=== Without CoT ===")
print(chat(user_message=problem))

print("\n=== With Zero-Shot CoT ===")
print(chat(user_message=problem + "\n\nLet's think step by step."))

### 5. ReAct Prompting (Reason + Act)

**ReAct** (Yao et al., 2022) interleaves reasoning traces with actions (e.g., tool calls). The model alternates between:
- **Thought:** Internal reasoning about what to do next
- **Action:** An operation to perform (web search, code execution, etc.)
- **Observation:** The result of the action

This is the foundation of modern **AI agents**. The example below shows the pattern using a simulated tool call:

In [ ]:
react_prompt = """Answer the following question using the Thought/Action/Observation pattern.
Available actions: Search[query], Calculate[expression], Finish[answer]

Question: What is the population of the capital of France, and what is it divided by 1000?

Thought: I need to find the capital of France and then look up its population.
Action: Search[capital of France]
Observation: Paris is the capital of France.
Thought: Now I need the population of Paris.
Action: Search[population of Paris]
Observation: Paris has a population of approximately 2,161,000.
Thought: Now I divide 2,161,000 by 1000.
Action: Calculate[2161000 / 1000]
Observation: 2161.0
Thought: I have the final answer.
Action: Finish[2161]"""

print(chat(user_message=react_prompt,
           system_message="You are a reasoning assistant that follows the ReAct pattern exactly."))

### Summary: When to Use Which Prompting Strategy

| Strategy | Best for | Limitations |
|---|---|---|
| Zero-shot | Common, well-defined tasks | May fail on unusual tasks |
| Few-shot | Novel tasks needing format guidance | Uses context window space |
| System prompt | Persona / constraint setting | Model may not always obey |
| Chain-of-Thought | Multi-step reasoning, math | Longer outputs; slower |
| ReAct | Agentic tasks with tools | Requires tool infrastructure |

For more techniques: https://www.promptingguide.ai/techniques

---

## Loading the Fine-Tuning Dataset

We use **OpenAssistant Guanaco** — the dataset from the original QLoRA paper. It contains ~10k human-written conversational exchanges in English, already formatted with the `### Human:` / `### Assistant:` template that instruction-tuned models expect.

Each example looks like:
```
### Human: What is the capital of France?
### Assistant: The capital of France is Paris.
```

In a real project you would inspect:
1. **Size** — is there enough data? LoRA fine-tuning can work with as few as a few hundred examples.
2. **Format** — does the text field follow a consistent chat template?
3. **Quality** — are responses the kind of behaviour you want to reinforce?

In [ ]:
from datasets import load_dataset

dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Features: {dataset.features}")
print("\nFirst example:")
print(dataset["text"][0])

## Parameter-Efficient Fine-Tuning (PEFT)

Full fine-tuning updates **all** model parameters. For a 7B model with the Adam optimiser, this requires:
- **Weights**: ~14 GB (bfloat16)
- **Gradients**: ~14 GB
- **Optimizer states** (Adam keeps 1st and 2nd moments): ~56 GB
- **Total**: ~84 GB — far beyond a typical research GPU

**PEFT** methods freeze most weights and only train a small number of additional parameters, reducing memory requirements by 10–100×.

---

## Low-Rank Adaptation (LoRA)

LoRA is the most widely used PEFT method. The core idea is elegant:

### The Mathematical Intuition

During fine-tuning, the *change* in a weight matrix $\Delta W$ tends to have a **low intrinsic rank** — meaning most of the useful update can be expressed in a much lower-dimensional subspace. LoRA exploits this by:

$$W' = W + \Delta W = W + \frac{\alpha}{r} \cdot B A$$

Where:
- $W \in \mathbb{R}^{d \times k}$ — the frozen original weight matrix
- $A \in \mathbb{R}^{r \times k}$ — a trainable down-projection (randomly initialised)
- $B \in \mathbb{R}^{d \times r}$ — a trainable up-projection (initialised to zeros, so $\Delta W = 0$ at start)
- $r$ — the **rank** (typically 4–64, much smaller than $d$ or $k$)
- $\alpha$ — a scaling hyperparameter

The original weights $W$ are frozen. Only $A$ and $B$ are trained.

### Parameter Savings

For a weight matrix of shape $d \times k$:
- Original parameters: $d \times k$
- LoRA parameters: $r \times k + d \times r = r(d + k)$
- Compression ratio: $\frac{dk}{r(d+k)}$ — typically **10–100×** fewer trainable params

![LoRA diagram](LoRA.png)

### Which Layers to Apply LoRA To?

LoRA can be applied to any linear layer in the transformer. Common choices:

| Layers | Notes |
|---|---|
| `q_proj, v_proj` | Original LoRA paper default — attention query and value projections |
| `q_proj, k_proj, v_proj, o_proj` | All attention projections — better quality, more params |
| `gate_proj, up_proj, down_proj` | Feed-forward layers — important for knowledge tasks |
| All linear layers | Maximum quality, approaches full fine-tuning |

The `target_modules` parameter in `LoraConfig` controls this.

### LoRA Applied to Transformer Blocks

![LoRA Transformer](LoRATransformer.png)

### LoRA Hyperparameters

- **`r` (rank)**: Dimension of the low-rank matrices. Higher rank = more parameters = better expressivity but more memory. Typical range: 8–64.
- **`lora_alpha`**: Scaling factor applied to $BA$. The effective scale is `alpha/r`. Setting `alpha = 2*r` is a common heuristic.
- **`lora_dropout`**: Dropout applied to the LoRA activations. Helps prevent overfitting on small datasets.
- **`bias`**: Whether to train bias terms. Usually left as `"none"` for memory efficiency.
- **`target_modules`**: Which linear layers to apply LoRA to.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,                   # Rank: balance between expressivity and parameter count
    lora_alpha=32,          # Scale factor (alpha/r = 2, a common starting point)
    lora_dropout=0.1,       # Dropout on LoRA activations
    bias="none",            # Don't train bias terms
    task_type="CAUSAL_LM",  # Task type for automatic layer selection logic
    # For Llama-3, target all attention projections:
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
def print_trainable_parameters(model):
    """Report the number of trainable vs total parameters."""
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(
        f"Trainable: {trainable_params:,}  |  "
        f"Total: {all_params:,}  |  "
        f"Trainable %: {100 * trainable_params / all_params:.4f}%"
    )

In [ ]:
# Inspect the model architecture before LoRA
print("=== Base model architecture (excerpt) ===")
print(model)

In [ ]:
# Wrap the quantised model with LoRA adapters
lora_model = get_peft_model(model, peft_config)

print("=== LoRA-wrapped model ===")
print(lora_model)
print()
print_trainable_parameters(lora_model)

Notice that:
- The original `Linear4bit` layers are unchanged and frozen
- Each targeted layer now has additional `lora_A` and `lora_B` sub-modules
- Only ~0.5–2% of parameters are trainable, drastically reducing memory and compute

## Configuring the Trainer

We use TRL's `SFTTrainer` with the modern `SFTConfig` (previously, training arguments were passed directly to `SFTTrainer`, which is now deprecated).

### Key Training Hyperparameters

| Parameter | Value | Explanation |
|---|---|---|
| `per_device_train_batch_size` | 1 | Single sample per GPU forward pass (memory constraint) |
| `gradient_accumulation_steps` | 4 | Effective batch size = 1×4 = 4 |
| `optim` | `paged_adamw_8bit` | AdamW with 8-bit optimizer states; paging offloads to CPU when GPU is full |
| `learning_rate` | 2e-4 | Typical for LoRA fine-tuning |
| `warmup_steps` | 20 | Linearly ramp the LR for the first 20 steps |
| `max_steps` | 500 | Total training steps (use `num_train_epochs` for epoch-based training) |
| `fp16` / `bf16` | bf16 | Mixed-precision training |
| `max_seq_length` | 512 | Truncate inputs longer than this |

### Gradient Accumulation

When batch size 1 is too small for stable training, **gradient accumulation** simulates a larger batch: gradients are summed over multiple forward passes before a single optimizer step. This is memory-free (no extra activations stored), only slightly slower.

### Paged Optimizer

The `paged_adamw_8bit` optimizer from bitsandbytes stores Adam's first and second moment buffers in 8-bit, and pages them to CPU RAM when GPU memory is full. This allows training models that would otherwise OOM.

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./results",
    # Batch size & gradient accumulation
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,          # Effective batch size = 4
    # Optimizer
    optim="paged_adamw_8bit",               # Memory-efficient AdamW
    # Learning rate schedule
    learning_rate=2e-4,
    lr_scheduler_type="cosine",             # Cosine decay (often better than constant)
    warmup_steps=20,                        # Gradual LR ramp at start
    # Training length
    max_steps=500,
    # Precision
    bf16=True,                              # Use bfloat16 (preferred over fp16 on modern GPUs)
    # Regularisation
    max_grad_norm=0.3,                      # Gradient clipping
    # Logging & saving
    logging_steps=10,
    save_steps=100,
    report_to="none",                       # Disable wandb/tensorboard for this demo
    # Dataset
    dataset_text_field="text",
    max_seq_length=512,
)

### Creating the SFTTrainer

`SFTTrainer` is a thin wrapper around `Trainer` that adds:
- Automatic instruction formatting (packing, chat template application)
- Integration with PEFT configs (automatically wraps the model in LoRA if `peft_config` is provided)
- Sequence packing for improved GPU utilisation

In [ ]:
from trl import SFTTrainer

# lora_model is already wrapped with LoRA from get_peft_model() above.
# Passing peft_config here would wrap it a second time — omit it.
trainer = SFTTrainer(
    model=lora_model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

## Training

Call `trainer.train()` to start. During training, watch the loss — it should decrease steadily. A typical training loss curve for SFT starts around 2.0–3.0 and converges to 0.5–1.5 depending on dataset and task complexity.

**Signs of problems:**
- Loss plateaus immediately → learning rate too low, or dataset formatting issue
- Loss explodes → learning rate too high, or gradient clipping too loose
- Loss reaches near-zero quickly → overfitting (dataset too small or too many steps)

In [ ]:
trainer.train()

## Saving the Adapter

A key advantage of LoRA is that you **only need to save the small adapter** (the $A$ and $B$ matrices), not the full model. This makes it easy to:
- Share adapters with collaborators (a few hundred MB instead of tens of GB)
- Swap adapters on top of the same base model for different tasks
- Publish adapters to the HuggingFace Hub

![LoRA Matrix](LoraMatrix.png)

In [ ]:
# Save only the LoRA adapter weights (NOT the full base model)
model_to_save = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
model_to_save.save_pretrained("outputs")

print("Adapter saved to ./outputs")

## Using the Fine-Tuned Model

### Same session
After training, `trainer.model` is already the PEFT-wrapped fine-tuned model — use it directly.

### New session (loading from disk)
If you are starting a fresh Python process, reload the base model and apply the saved adapter with `PeftModel.from_pretrained`. Do **not** call `get_peft_model()` again — that wraps an already-wrapped model and creates duplicate adapters.

In [ ]:
from peft import PeftModel

# --- Same session ---
# trainer.model is already fine-tuned; use it directly.
finetuned_model = trainer.model

# --- New session (run this block instead if starting fresh) ---
# run the code cell below.


In [4]:
# --- New session (run this block instead if starting fresh) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Store weights in 4-bit
    bnb_4bit_quant_type="nf4",      # Use NormalFloat 4-bit (optimal for normally-distributed weights)
    bnb_4bit_use_double_quant=True, # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for actual matrix multiplications
)

base = AutoModelForCausalLM.from_pretrained(
     model_name, quantization_config=bnb_config, device_map='auto'
     )

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Llama-3 does not have a dedicated pad token — use eos_token as pad
# padding_side='right' avoids overflow issues in half-precision training
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(base, 'outputs')

from datasets import load_dataset

dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Features: {dataset.features}")
print("\nFirst example:")
print(dataset["text"][0])

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Dataset size: 9846 examples
Features: {'text': Value('string')}

First example:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.

Recent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, worke

In [2]:
# Test the fine-tuned model on a sample prompt from the dataset
prompt = dataset["text"][0].split("### Assistant:")[0] + "### Assistant:"
print("Prompt:")
print(prompt)

device = "cuda:0"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = finetuned_model.generate(**inputs, max_new_tokens=300, do_sample=True, temperature=0.7)

print("\nGenerated:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant:


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: A monopsony is a market structure in which there is only one buyer of a good or service. In the context of the labor market, a monopsony occurs when there is only one employer that is willing to hire a particular type of worker.

The existence of monopsonies in the labor market is relevant because it can lead to lower wages for workers. Since there is only one employer willing to hire a particular type of worker, the employer has bargaining power and can negotiate lower wages. This is because the worker is not able to negotiate with other employers for better wages, as there are no other employers willing to hire them.

Research has shown that monopsonies are more common than previously thought. A study by Autor, E. (2019) found that monopsonies are present 

### Prompting the Fine-Tuned Model with a Custom Input

The Guanaco dataset uses the `### Human: ... ### Assistant:` template. To prompt the fine-tuned model correctly, we format our own question in the same template and let the model complete the `### Assistant:` part.

In [3]:
# Edit this prompt freely — just keep the ### Human / ### Assistant structure
user_prompt = "Explain the difference between supervised and unsupervised learning."

# Format using the Guanaco template
prompt = f"### Human: {user_prompt}\n### Assistant:"

device = "cuda:0"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.1,
    )

# Decode and strip the prompt from the output
full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = full_output[len(prompt):].strip()
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Supervised learning is a type of machine learning where the algorithm is trained on labeled data, meaning that each sample in the training dataset has an associated label or target output. The goal of supervised learning is to learn a mapping from input data to output labels, so that the algorithm can make accurate predictions on new, unseen data.

On the other hand, unsupervised learning involves training an algorithm on unlabeled data, without any target outputs. The goal of unsupervised learning is to identify patterns or structure within the data itself, such as clustering similar samples together or identifying anomalies. Unsupervised learning algorithms do not have a specific target output to predict, but rather try to learn representations of the data that capture useful information about its underlying distribution.

In summary, supervised learning is used for classification and regression tasks, while unsupervised learning is used for exploratory data analysis, clustering, and

# Merge model and adapter

In [5]:
from peft import PeftModel

# Load base model in full precision for merging.
# Use device_map={'': 0} (pin to GPU 0) instead of 'auto' — device_map='auto'
# triggers a bug in accelerate's get_balanced_memory when used with PeftModel.
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={'': 0},
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Apply the saved adapter on top of the full-precision base
peft_model = PeftModel.from_pretrained(base_model, 'outputs', device_map={'': 0})

# Merge adapter weights into the base model and remove adapter modules
merged_model = peft_model.merge_and_unload()

# Save as a standard HuggingFace model (no adapter infrastructure)
merged_model.save_pretrained('outputs_merged')
tokenizer.save_pretrained('outputs_merged')

print('Merged model saved to ./outputs_merged')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to ./outputs_merged


## Summary

In this notebook we covered:

### Prompting
| Technique | Key idea |
|---|---|
| Zero-shot | No examples, rely on pre-trained knowledge |
| Few-shot | Examples in the prompt for in-context learning |
| System prompts | Persona and constraint setting via the system role |
| Chain-of-Thought | Force step-by-step reasoning for complex tasks |
| ReAct | Interleave reasoning and actions for agentic tasks |

### Fine-Tuning
| Concept | Key idea |
|---|---|
| QLoRA | 4-bit quantisation + LoRA for memory-efficient fine-tuning |
| LoRA | Train only low-rank update matrices; freeze original weights |
| SFTTrainer | HuggingFace TRL trainer for instruction fine-tuning |
| Adapter saving | Save only the small adapter, not the full model |
| Merging | Fold the adapter back into base weights for faster inference |

---

## References

- **QLoRA** — Dettmers et al. (2023): https://arxiv.org/abs/2305.14314
- **LoRA** — Hu et al. (2021): https://arxiv.org/abs/2106.09685
- **Chain-of-Thought** — Wei et al. (2022): https://arxiv.org/abs/2201.11903
- **ReAct** — Yao et al. (2022): https://arxiv.org/abs/2210.03629
- **4-bit quantisation blog** — HuggingFace: https://huggingface.co/blog/4bit-transformers-bitsandbytes
- **PEFT library** — https://huggingface.co/docs/peft/en/index
- **TRL SFTTrainer** — https://huggingface.co/docs/trl/main/en/sft_trainer
- **Prompting guide** — https://www.promptingguide.ai/techniques
- **LoRA explained** — https://towardsdatascience.com/lora-intuitively-and-exhaustively-explained-e944a6bff46b